# Advanced workflows and diagnostics

The earlier tutorials cover the two most common tasks: predicting an oscillation range and estimating stellar properties from measured global oscillations. This notebook collects the less frequently needed features: requesting every available quantity, choosing how uncertain inputs are interpreted, controlling relation scatter, replacing a prior, checking calibration validity, inspecting raw sampler results, and solving several stars in a batch.

These controls do not make weak data informative. Whenever a posterior is broad, multimodal, or strongly prior-sensitive, report that limitation rather than selecting only the most convenient summary.

In [ ]:
import numpy as np
import asteroscale as ast
from asteroscale.distributions import TruncatedNormal

## Request every available quantity

`want="all"` expands to every fundamental and derived name. A forward calculation needs all six fundamentals if every registered relation is to be evaluated. The Gaia photometric outputs in this example are approximate placeholders, not precision synthetic photometry.

In [ ]:
solar_like_star = {
    "M": 1.0,       # solar masses
    "R": 1.0,       # solar radii
    "Teff": 5772.0, # K
    "plx": 10.0,    # mas
    "A_G": 0.05,    # mag
    "FeH": 0.0,     # dex
}
all_quantities = ast.solve(solar_like_star, want="all")
for name, value in all_quantities.items():
    print(f"{name:12s} {value:10.4f}")

The {doc}`../reference/quantities` page gives units and dependencies. In most analyses, requesting only the quantities you intend to interpret keeps the result easier to audit.

## Propagation versus Bayesian conditioning

An uncertain fundamental input can mean either "this distribution is my current knowledge" or "this is a new measurement to combine with a population prior". `propagate` implements the first interpretation and `likelihood` the second. Derived observables such as `numax` and `dnu` are likelihood terms in both modes.

In [ ]:
seismic_data = {
    "Teff": (5777.0, 80.0),
    "FeH": (0.0, 0.10),
    "numax": (3090.0, 30.0),
    "dnu": (135.1, 1.0),
}

propagated = ast.Solver(
    preset="fast", input_mode="propagate", seed=10
).solve(seismic_data, want=["M", "R"])

conditioned = ast.Solver(
    preset="fast", input_mode="likelihood", seed=10
).solve(seismic_data, want=["M", "R"])

print("Propagated input distributions")
ast.summarize(propagated)
print("\nMeasurements conditioned on field-star priors")
ast.summarize(conditioned)

The answers can be similar for precise measurements. Differences become more important when the measurements are weak or conflict with a population prior. Do not use `likelihood` on a catalogue posterior without considering whether that would apply its original prior twice.

## Relation scatter is part of the model

`fast` and `standard` disable empirical relation scatter by default. `precise` enables the calibrated defaults as well as using more live points and a tighter stopping criterion. To separate the effect of relation scatter from the effect of a longer run, compare precise calculations with and without scatter.

In [ ]:
precise_with_scatter = ast.Solver(
    preset="precise", seed=20
).solve(seismic_data, want=["M", "R"])

precise_without_scatter = ast.Solver(
    preset="precise", relation_scatter=0.0, seed=20
).solve(seismic_data, want=["M", "R"])

ast.summarize(precise_with_scatter)
ast.summarize(precise_without_scatter)

Because scatter broadens the compatible mass-radius combinations, the priors gain influence and the posterior median can move. This is not merely numerical noise. See {doc}`../concepts/calibration-and-scatter` before interpreting the comparison.

## Replace a default prior

A custom prior must provide a `ppf(u)` method so Dynesty can transform a unit-cube coordinate. If the same object is used as a likelihood, it must also provide `logpdf(x)`. Here an external analysis motivates a narrower, truncated mass prior.

In [ ]:
external_mass_prior = TruncatedNormal(
    loc=1.05, scale=0.10, low=0.7, high=1.4
)
custom_solver = ast.Solver(
    priors={"M": external_mass_prior},
    input_mode="likelihood",
    preset="fast",
    seed=30,
)
custom_samples = custom_solver.solve(
    seismic_data, want=["M", "R"]
)
ast.summarize(custom_samples)

A tighter prior is not automatically a better prior. Record its origin and test whether the scientific conclusion changes under reasonable alternatives.

## Inspect calibration validity and raw Dynesty output

A validity report identifies extrapolation beyond adopted calibration domains. `return_results=True` additionally exposes Dynesty's weighted samples, log weights, evidence estimate, and diagnostics for users who need them.

In [ ]:
diagnostic_solver = ast.Solver(preset="fast", seed=40)
diagnostic = diagnostic_solver.solve(
    seismic_data,
    want=["M", "R"],
    return_validity=True,
    return_results=True,
)

print(diagnostic["_validity"])
raw = diagnostic["_results"]
weights = np.exp(raw.logwt - raw.logz[-1])
print("Nested samples:", len(raw.samples))
print("Effective weighted sample count:", 1.0 / np.sum(weights**2))

The ordinary `M` and `R` arrays returned by `solve` are equal-weight posterior samples and are usually the right inputs to `summarize` and `plot_posterior`. The raw result is mainly for convergence checks and custom weighted plots.

## Solve several independent stars

`solve_many` distributes independent targets across worker processes. Use a fixed `base_seed` for reproducibility. In restricted notebook environments, multiprocessing may be unavailable; the same call works from a normal Python script.

In [ ]:
targets = {
    "solar analogue": seismic_data,
    "subgiant": {
        "Teff": (5600.0, 80.0),
        "FeH": (0.0, 0.10),
        "numax": (1000.0, 30.0),
        "dnu": (57.0, 0.8),
    },
}
batch = ast.solve_many(
    targets,
    want=["M", "R"],
    preset="fast",
    n_jobs=2,
    base_seed=100,
)
for target, result in batch.items():
    print("\n", target)
    ast.summarize(result)

## A practical checking sequence

Before trusting a result: (1) check names and units; (2) run `fast`; (3) inspect one- and two-dimensional posterior plots; (4) compare with a different seed; (5) examine prior sensitivity and validity warnings; and (6) use `precise` only after deciding whether relation scatter belongs in the uncertainty model. Agreement between seeds supports numerical stability, but it does not validate the empirical scaling relation itself.